# Load COCO Reference Images and Libraries

In [ ]:
!pip install -q transformers torch torchvision pillow accelerate huggingface-hub requests torch-fidelity torchmetrics
!pip show torch transformers accelerate sentencepiece

# Clone Faster-Diffusion repo
import os
if not os.path.exists('Faster-Diffusion'):
    !git clone https://github.com/hutaiHang/Faster-Diffusion.git
else:
    print('Faster-Diffusion repo already cloned')

In [ ]:
import sys
sys.path.insert(0, './Faster-Diffusion')

import os
import json
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from diffusers import StableDiffusionPipeline, DiffusionPipeline
from torch_fidelity import calculate_metrics
from utils import register_parallel_pipeline, register_faster_forward
from PIL import Image
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel

In [ ]:
!pip freeze > requirements.txt

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

BATCH_SIZE = 8 if DEVICE == "cuda" else 2
NUM_STEPS = 25
NUM_IMAGES = 1000   # important for stable FID

OUTPUT_ROOT = "outputs"
COCO_DIR = "coco_real"

In [ ]:
!wget http://images.cocodataset.org/zips/val2017.zip
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip

!unzip val2017.zip
!unzip annotations_trainval2017.zip

In [ ]:
COCO_IMG_DIR = "val2017"
COCO_ANN_FILE = "annotations/captions_val2017.json"


def load_coco_val2017():
    with open(COCO_ANN_FILE, "r") as f:
        data = json.load(f)

    # Map image_id -> captions
    captions_dict = {}
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        captions_dict.setdefault(img_id, []).append(ann["caption"])

    # Map image_id -> filename
    id_to_file = {img["id"]: img["file_name"] for img in data["images"]}

    samples = []
    for img_id, file_name in id_to_file.items():
        path = os.path.join(COCO_IMG_DIR, file_name)
        captions = captions_dict.get(img_id, [])

        if os.path.exists(path) and captions:
            samples.append({
                "image_path": path,
                "caption": captions[0]  # pick first caption
            })

    return samples


samples = load_coco_val2017()
print(f"Loaded {len(samples)} samples")

In [ ]:
OUTPUT_REAL = "coco_real"
os.makedirs(OUTPUT_REAL, exist_ok=True)

for i, sample in enumerate(samples):
    img = Image.open(sample["image_path"]).convert("RGB")
    img = img.resize((512, 512))
    img.save(f"{OUTPUT_REAL}/{i}.png")

print("Saved resized COCO images")

### Faster-Diffusion Computation w/ FID Metric

In [ ]:
def load_prompts(samples, num_images):
    return [s["caption"] for s in samples[:num_images]]


prompts = load_prompts(samples, NUM_IMAGES)

def load_pipeline(name):
    if name == "sd1.5":
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=DTYPE
        )
    elif name == "sd2.1":
        pipe = DiffusionPipeline.from_pretrained(
            "sd2-community/stable-diffusion-2-1",
            torch_dtype=DTYPE
        )

    pipe = pipe.to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

def enable_faster_diffusion(pipe, use_faster):
    if not use_faster:
        return
    register_parallel_pipeline(pipe)
    register_faster_forward(pipe.unet, mod='50ls')

In [ ]:
def generate_images(pipe, prompts, out_dir, use_faster):
    os.makedirs(out_dir, exist_ok=True)

    enable_faster_diffusion(pipe, use_faster)

    all_times = []

    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        batch_size = len(batch_prompts)

        # SAME LATENTS for fairness
        latents = torch.randn(
            (batch_size, pipe.unet.in_channels, 64, 64),
            device=DEVICE,
            dtype=DTYPE
        )

        start = time.time()

        images = pipe(
            batch_prompts,
            num_inference_steps=NUM_STEPS,
            latents=latents
        ).images

        end = time.time()
        all_times.append(end - start)

        for j, img in enumerate(images):
            img.save(f"{out_dir}/{i+j}.png")

    return np.mean(all_times), np.std(all_times)

def compute_fid(real_dir, fake_dir):
    metrics = calculate_metrics(
        input1=real_dir,
        input2=fake_dir,
        fid=True,
        isc=False,
        kid=False,
        cuda=(DEVICE == "cuda")
    )
    return metrics["frechet_inception_distance"]

In [ ]:
NUM_IMAGES = len(samples)
prompts = load_prompts(samples, NUM_IMAGES)

configs = [
    ("sd1.5", False),
    ("sd1.5", True),
    ("sd2.1", False),
    ("sd2.1", True),
]

results = {}

for model, faster in configs:
    print(f"\nRunning {model} | faster={faster}")

    pipe = load_pipeline(model)

    out_dir = f"{OUTPUT_ROOT}/{model}_{'faster' if faster else 'base'}"

    mean_t, std_t = generate_images(pipe, prompts, out_dir, faster)

    results[(model, faster)] = {
        "dir": out_dir,
        "time_mean": mean_t,
        "time_std": std_t
    }

    del pipe
    torch.cuda.empty_cache()

In [ ]:
fid_results = {}

for model in ["sd1.5", "sd2.1"]:
    base_dir = results[(model, False)]["dir"]
    faster_dir = results[(model, True)]["dir"]

    fid_base = compute_fid(COCO_DIR, base_dir)
    fid_faster = compute_fid(COCO_DIR, faster_dir)

    fid_results[model] = {
        "base": fid_base,
        "faster": fid_faster,
        "delta": fid_faster - fid_base
    }

print("\n=== FINAL RESULTS ===")

for (model, faster), data in results.items():
    print(f"{model} | faster={faster}")
    print(f"  Time: {data['time_mean']:.3f} \u00b1 {data['time_std']:.3f}")

for model, vals in fid_results.items():
    print(f"\n{model} FID:")
    print(f"  Base:   {vals['base']:.3f}")
    print(f"  Faster: {vals['faster']:.3f}")
    print(f"  \u0394FID:   {vals['delta']:.3f}")

### CLIP Computation

In [ ]:
def load_clip():
    model = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32"
    ).to(DEVICE)

    processor = CLIPProcessor.from_pretrained(
        "openai/clip-vit-base-patch32"
    )

    model.eval()
    return model, processor

In [ ]:
def compute_clip_score(image_dir, prompts, model, processor):
    image_files = sorted(os.listdir(image_dir))

    scores = []

    for i in tqdm(range(0, len(image_files), BATCH_SIZE)):
        batch_files = image_files[i:i+BATCH_SIZE]
        batch_images = []
        batch_texts = []

        for j, file in enumerate(batch_files):
            img = Image.open(os.path.join(image_dir, file)).convert("RGB")
            batch_images.append(img)

            # match prompt index
            idx = i + j
            batch_texts.append(prompts[idx])

        inputs = processor(
            text=batch_texts,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            # Normalize (CRITICAL)
            image_embeds = F.normalize(image_embeds, dim=-1)
            text_embeds = F.normalize(text_embeds, dim=-1)

            # cosine similarity
            batch_scores = (image_embeds * text_embeds).sum(dim=-1)

        scores.extend(batch_scores.cpu().numpy())

    return float(np.mean(scores))

In [ ]:
clip_model, clip_processor = load_clip()

clip_results = {}

for model in ["sd1.5", "sd2.1"]:
    base_dir = results[(model, False)]["dir"]
    faster_dir = results[(model, True)]["dir"]

    print(f"\nComputing CLIP for {model}...")

    clip_base = compute_clip_score(
        base_dir, prompts, clip_model, clip_processor
    )

    clip_faster = compute_clip_score(
        faster_dir, prompts, clip_model, clip_processor
    )

    clip_results[model] = {
        "base": clip_base,
        "faster": clip_faster,
        "delta": clip_faster - clip_base
    }

# Results

In [ ]:
for model, vals in clip_results.items():
    print(f"\n{model} CLIP:")
    print(f"  Base:   {vals['base']:.4f}")
    print(f"  Faster: {vals['faster']:.4f}")
    print(f"  \u0394CLIP:  {vals['delta']:.4f}")

In [ ]:
# Print full timing + metric summary matching DeepCache output format
print("\n=== FULL SUMMARY ===")
for (model, faster), data in results.items():
    print(f"{model} | faster={faster}")
    print(f"  Time: {data['time_mean']:.3f} \u00b1 {data['time_std']:.3f}")

for model in ["sd1.5", "sd2.1"]:
    print(f"\n{model} FID:")
    print(f"  Base:   {fid_results[model]['base']:.3f}")
    print(f"  Faster: {fid_results[model]['faster']:.3f}")
    print(f"  \u0394FID:   {fid_results[model]['delta']:.3f}")
    print(f"\n{model} CLIP:")
    print(f"  Base:   {clip_results[model]['base']:.4f}")
    print(f"  Faster: {clip_results[model]['faster']:.4f}")
    print(f"  \u0394CLIP:  {clip_results[model]['delta']:.4f}")

| Model | Speedup | FID ↑ | CLIP Δ |
| ----- | ------- | ----- | ------ |
| SD1.5 | ?×      | ?     | ?      |
| SD2.1 | ?×      | ?     | ?      |

| Faster-Diffusion Model Name     | Training Time | Mean Time  | Var. Time  | FID Score | CLIP Score |
| ------------------------------- | ------------- | ---------- | ---------- | --------- | ---------- |
| SD1.5, Faster=False             | ?             | ? secs     | ? secs     | ?         | ?          |
| SD1.5, Faster=True  (mod=50ls)  | ?             | ? secs     | ? secs     | ?         | ?          |
| SD2.1, Faster=False             | ?             | ? secs     | ? secs     | ?         | ?          |
| SD2.1, Faster=True  (mod=50ls)  | ?             | ? secs     | ? secs     | ?         | ?          |